In [15]:
## Data ingestion
# !pip install beautifulsoup4
!pip install -U langchain langchain-core langchain-community

In [16]:
import os 
from dotenv import load_dotenv
load_dotenv()

# OpenAI API Key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")



In [ ]:

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma:2b",
    temperature=0.7
)
print(llm)

model='gemma:2b' temperature=0.7


In [18]:
#Data ingestion - from the website we need to scrape the data 

from langchain_community.document_loaders import WebBaseLoader

In [19]:
# WebBaseLoader is a library that takes urls and scrapes the data from those urls and gives us the data in a format that we can use for our langchain applications. It uses beautifulsoup4 under the hood to scrape the data from the websites.
loader = WebBaseLoader("https://www.geeksforgeeks.org/dsa/introduction-to-doubly-linked-lists-in-java/")
loader

In [20]:
docs=loader.load()
docs[0]

Document(metadata={'source': 'https://www.geeksforgeeks.org/dsa/introduction-to-doubly-linked-lists-in-java/', 'title': 'Introduction to Doubly Linked Lists in Java - GeeksforGeeks', 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.', 'language': 'en'}, page_content='Introduction to Doubly Linked Lists in Java - GeeksforGeeksCoursesTutorialsPracticeJobsDSA TutorialInterview QuestionsQuizzesMust DoAdvanced DSASystem DesignAptitudePuzzlesInterview CornerDSA PythonShare Your ExperiencesDSA FundamentalsLogic Building ProblemsAnalysis of AlgorithmsData StructuresArray Data StructureString in Data StructureHashing in Data StructureLinked List Data StructureStack Data StructureQueue Data StructureTree Data StructureGraph Data StructureTrie Data StructureAlgorithmsSearching Algo

In [21]:
#Load Data => Docs => Divide our text into smaller chunks => Into Texts => Turn into vectors => Vector Embeddings => Vector Store DB .   
from langchain_text_splitters import RecursiveCharacterTextSplitter
textSplitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents=textSplitter.split_documents(docs)

In [22]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="gemma:2b"
)

embeddings

OllamaEmbeddings(model='gemma:2b', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [23]:
from langchain_community.vectorstores import FAISS
vectorStoredb = FAISS.from_documents(documents, embeddings)

In [24]:
vectorStoredb

In [25]:
query="How Insertion in a Doubly Linked List in Java is done ?"
result = vectorStoredb.similarity_search(query)
result[0].page_content

'Doubly linked list is a data structure that has reference to both the previous and next nodes in the list. It provides simplicity to traverse, insert and delete the nodes in both directions in a list.'

In [28]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context:

<context>
{context}
</context>

Question: {input}
""")

document_chain = prompt | llm | StrOutputParser() 
document_chain

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n\n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwargs={})])
| ChatOllama(model='gemma:2b', temperature=0.7)
| StrOutputParser()

In [29]:
document_chain.invoke({
    "context": result[0].page_content,
    "input": "what is doubely linked list ?"
})

'Sure. A doubly linked list is a data structure that has reference to both the previous and next nodes in the list. It provides simplicity to traverse, insert and delete the nodes in both directions in a list.'

In [35]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vectorStoredb.as_retriever()

retrieval_chain = (
    {
        "context": retriever,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)
retrieval_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000015EDB93AC60>, search_kwargs={}),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n\n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwargs={})])
| ChatOllama(model='gemma:2b', temperature=0.7)
| StrOutputParser()

In [41]:
# Get the response from the llm 

response = retrieval_chain.invoke(" what is next in doubely linked list ?")


In [42]:
response

'The context does not specify what is next in a doubly linked list, so I cannot answer this question from the provided context.'